# Whisper 한국어 도메인 파인튜닝

## 목표
개발 용어 인식률 향상 (PR, 머지, 커밋, API, 도커 등)

## 환경
- **서버**: SSAFY GPU 서버 (A100, Device3)
- **모델**: openai/whisper-large-v3
- **방법**: LoRA 파인튜닝

## 실험 구조
```
1. [Before] 베이스 모델 테스트 → WER 측정
2. [Training] LoRA 파인튜닝
3. [After] 파인튜닝 모델 테스트 → WER 측정
4. [Compare] 결과 비교표
```

---
# Part 0: 환경 설정

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"  # Device3 지정

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
!pip install -q transformers datasets accelerate peft bitsandbytes
!pip install -q librosa soundfile jiwer evaluate
!pip install -q tensorboard pandas

## 테스트 문장 정의 (개발 용어 포함)

In [ ]:
# 개발 용어가 포함된 테스트 문장들
TEST_SENTENCES = [
    "PR 올렸으니까 리뷰 부탁드려요",
    "머지 컨플릭트 해결해야 해요",
    "API 엔드포인트 수정했습니다",
    "도커 컨테이너 재시작해주세요",
    "CI CD 파이프라인 확인해봐",
    "깃허브에 커밋 푸시했어요",
    "백엔드 서버 배포 완료됐습니다",
    "리팩토링 브랜치 만들었어요",
    "스프린트 회고 시작하겠습니다",
    "코드 리뷰 코멘트 남겼어요"
]

# 결과 저장용
results = {
    "sentence": TEST_SENTENCES,
    "before_prediction": [],
    "after_prediction": [],
    "before_wer": [],
    "after_wer": []
}

---
# Part 1: [BEFORE] 베이스 모델 테스트

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import evaluate

MODEL_NAME = "openai/whisper-large-v3"

print("[BEFORE] 베이스 모델 로딩...")
processor = WhisperProcessor.from_pretrained(MODEL_NAME)
base_model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)
base_model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="ko", task="transcribe")

wer_metric = evaluate.load("wer")
print("베이스 모델 로딩 완료!")

### TTS로 테스트 오디오 생성 (또는 실제 녹음 파일 사용)

In [ ]:
# 옵션 1: gTTS로 테스트 오디오 생성
!pip install -q gtts pydub

from gtts import gTTS
from pydub import AudioSegment
import io

os.makedirs("test_audio", exist_ok=True)

def generate_test_audio(text, idx):
    """TTS로 테스트 오디오 생성"""
    tts = gTTS(text=text, lang='ko')
    mp3_path = f"test_audio/test_{idx}.mp3"
    wav_path = f"test_audio/test_{idx}.wav"
    
    tts.save(mp3_path)
    
    # mp3 -> wav 변환 (16kHz)
    audio = AudioSegment.from_mp3(mp3_path)
    audio = audio.set_frame_rate(16000).set_channels(1)
    audio.export(wav_path, format="wav")
    
    return wav_path

# 테스트 오디오 생성
test_audio_paths = []
for idx, sentence in enumerate(TEST_SENTENCES):
    path = generate_test_audio(sentence, idx)
    test_audio_paths.append(path)
    print(f"생성: {path}")

print(f"\n총 {len(test_audio_paths)}개 테스트 오디오 생성 완료")

### 베이스 모델로 테스트

In [ ]:
import librosa

def transcribe(model, audio_path):
    """오디오 -> 텍스트"""
    audio, sr = librosa.load(audio_path, sr=16000)
    inputs = processor(audio, sampling_rate=16000, return_tensors="pt").input_features
    inputs = inputs.to(model.device, dtype=torch.float16)
    
    with torch.no_grad():
        predicted_ids = base_model.generate(inputs)
    
    return processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

print("=" * 60)
print("[BEFORE] 베이스 모델 테스트 결과")
print("=" * 60)

for idx, (audio_path, ground_truth) in enumerate(zip(test_audio_paths, TEST_SENTENCES)):
    prediction = transcribe(base_model, audio_path)
    wer = wer_metric.compute(predictions=[prediction], references=[ground_truth])
    
    results["before_prediction"].append(prediction)
    results["before_wer"].append(wer)
    
    print(f"\n[{idx+1}] 정답: {ground_truth}")
    print(f"    예측: {prediction}")
    print(f"    WER:  {wer:.2%}")

avg_before_wer = sum(results["before_wer"]) / len(results["before_wer"])
print(f"\n{'='*60}")
print(f"[BEFORE] 평균 WER: {avg_before_wer:.2%}")
print(f"{'='*60}")

In [ ]:
# 메모리 정리
del base_model
torch.cuda.empty_cache()
print("베이스 모델 메모리 해제 완료")

---
# Part 2: [TRAINING] LoRA 파인튜닝

## 2.1 학습 데이터 준비

In [ ]:
from datasets import load_dataset, Audio, Dataset

# 옵션 A: 공개 데이터셋 (테스트용)
print("학습 데이터 로딩...")
train_dataset = load_dataset(
    "mozilla-foundation/common_voice_11_0", 
    "ko", 
    split="train[:2000]",  # 2000개 샘플
    trust_remote_code=True
)
train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16000))

eval_dataset = load_dataset(
    "mozilla-foundation/common_voice_11_0", 
    "ko", 
    split="validation[:200]",
    trust_remote_code=True
)
eval_dataset = eval_dataset.cast_column("audio", Audio(sampling_rate=16000))

print(f"학습 데이터: {len(train_dataset)}개")
print(f"검증 데이터: {len(eval_dataset)}개")

In [ ]:
# 옵션 B: AI Hub 데이터 (실제 사용 권장)
# AI Hub에서 다운로드: https://aihub.or.kr
# - "한국어 음성" 또는 "자유대화 음성" 데이터셋

AIHUB_PATH = "/home/i14d106/data/aihub/"  # 실제 경로로 변경

def load_aihub_dataset(data_path, max_samples=2000):
    """AI Hub 데이터 로드"""
    import json
    import glob
    
    samples = []
    audio_files = glob.glob(f"{data_path}/**/*.wav", recursive=True)[:max_samples]
    
    for audio_path in audio_files:
        json_path = audio_path.replace(".wav", ".json")
        if os.path.exists(json_path):
            with open(json_path, "r", encoding="utf-8") as f:
                label = json.load(f)
            samples.append({
                "audio": audio_path,
                "sentence": label.get("transcription", label.get("text", ""))
            })
    
    return Dataset.from_list(samples)

# AI Hub 데이터가 있으면 사용
# train_dataset = load_aihub_dataset(AIHUB_PATH)

## 2.2 데이터 전처리

In [ ]:
# 모델 다시 로드 (학습용)
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

def prepare_dataset(batch):
    """데이터 전처리"""
    audio = batch["audio"]
    
    batch["input_features"] = processor(
        audio["array"], 
        sampling_rate=audio["sampling_rate"],
        return_tensors="pt"
    ).input_features[0]
    
    batch["labels"] = processor.tokenizer(batch["sentence"]).input_ids
    return batch

print("데이터 전처리 중...")
train_dataset = train_dataset.map(prepare_dataset, remove_columns=train_dataset.column_names)
eval_dataset = eval_dataset.map(prepare_dataset, remove_columns=eval_dataset.column_names)
print("전처리 완료!")

## 2.3 LoRA 설정

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

print("\n[LoRA 설정]")
print(f"- Rank: {lora_config.r}")
print(f"- Alpha: {lora_config.lora_alpha}")
print(f"- Target: {lora_config.target_modules}")
model.print_trainable_parameters()

## 2.4 학습 실행

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(processor.tokenizer, model=model, padding=True)

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-korean-finetuned",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    warmup_steps=100,
    max_steps=1000,  # 테스트용, 실제로는 4000+
    fp16=True,
    evaluation_strategy="steps",
    eval_steps=200,
    save_steps=200,
    logging_steps=50,
    report_to="tensorboard",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

In [ ]:
print("=" * 60)
print("[TRAINING] 학습 시작")
print("=" * 60)
print(f"- 학습 데이터: {len(train_dataset)}개")
print(f"- 검증 데이터: {len(eval_dataset)}개")
print(f"- Batch size: {training_args.per_device_train_batch_size}")
print(f"- Max steps: {training_args.max_steps}")
print(f"- Learning rate: {training_args.learning_rate}")
print("=" * 60)

train_result = trainer.train()

print("\n[학습 완료]")
print(f"- Total steps: {train_result.global_step}")
print(f"- Training loss: {train_result.training_loss:.4f}")

## 2.5 모델 저장

In [ ]:
# LoRA 어댑터 저장
LORA_PATH = "./whisper-korean-lora"
MERGED_PATH = "./whisper-korean-merged"

model.save_pretrained(LORA_PATH)
print(f"LoRA 어댑터 저장: {LORA_PATH}")

# 병합 모델 저장
merged_model = model.merge_and_unload()
merged_model.save_pretrained(MERGED_PATH)
processor.save_pretrained(MERGED_PATH)
print(f"병합 모델 저장: {MERGED_PATH}")

---
# Part 3: [AFTER] 파인튜닝 모델 테스트

In [ ]:
# 메모리 정리 후 파인튜닝 모델 로드
del model, merged_model
torch.cuda.empty_cache()

print("[AFTER] 파인튜닝 모델 로딩...")
finetuned_model = WhisperForConditionalGeneration.from_pretrained(
    MERGED_PATH,
    torch_dtype=torch.float16,
    device_map="auto"
)
finetuned_model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="ko", task="transcribe")
print("파인튜닝 모델 로딩 완료!")

In [ ]:
def transcribe_finetuned(audio_path):
    """파인튜닝 모델로 추론"""
    audio, sr = librosa.load(audio_path, sr=16000)
    inputs = processor(audio, sampling_rate=16000, return_tensors="pt").input_features
    inputs = inputs.to(finetuned_model.device, dtype=torch.float16)
    
    with torch.no_grad():
        predicted_ids = finetuned_model.generate(inputs)
    
    return processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

print("=" * 60)
print("[AFTER] 파인튜닝 모델 테스트 결과")
print("=" * 60)

for idx, (audio_path, ground_truth) in enumerate(zip(test_audio_paths, TEST_SENTENCES)):
    prediction = transcribe_finetuned(audio_path)
    wer = wer_metric.compute(predictions=[prediction], references=[ground_truth])
    
    results["after_prediction"].append(prediction)
    results["after_wer"].append(wer)
    
    print(f"\n[{idx+1}] 정답: {ground_truth}")
    print(f"    예측: {prediction}")
    print(f"    WER:  {wer:.2%}")

avg_after_wer = sum(results["after_wer"]) / len(results["after_wer"])
print(f"\n{'='*60}")
print(f"[AFTER] 평균 WER: {avg_after_wer:.2%}")
print(f"{'='*60}")

---
# Part 4: [COMPARE] 결과 비교

In [ ]:
import pandas as pd

# 비교 테이블 생성
comparison_df = pd.DataFrame({
    "정답": results["sentence"],
    "Before 예측": results["before_prediction"],
    "After 예측": results["after_prediction"],
    "Before WER": [f"{w:.2%}" for w in results["before_wer"]],
    "After WER": [f"{w:.2%}" for w in results["after_wer"]],
    "개선율": [f"{(b-a)/b*100:.1f}%" if b > 0 else "N/A" 
              for b, a in zip(results["before_wer"], results["after_wer"])]
})

print("=" * 80)
print("[COMPARISON] Before vs After 결과 비교")
print("=" * 80)
print(comparison_df.to_string(index=False))

In [ ]:
# 요약 통계
print("\n" + "=" * 60)
print("[SUMMARY] 학습 결과 요약")
print("=" * 60)
print(f"\n📊 평균 WER 비교")
print(f"   Before (베이스 모델): {avg_before_wer:.2%}")
print(f"   After (파인튜닝):     {avg_after_wer:.2%}")
print(f"   개선율:               {(avg_before_wer - avg_after_wer) / avg_before_wer * 100:.1f}%")

print(f"\n📁 학습 정보")
print(f"   학습 데이터: {len(train_dataset)}개")
print(f"   학습 스텝: {training_args.max_steps}")
print(f"   LoRA Rank: {lora_config.r}")

print(f"\n💾 저장 경로")
print(f"   LoRA 어댑터: {LORA_PATH}")
print(f"   병합 모델: {MERGED_PATH}")
print("=" * 60)

In [ ]:
# 결과 CSV 저장
comparison_df.to_csv("whisper_finetune_results.csv", index=False, encoding="utf-8-sig")
print("결과 저장: whisper_finetune_results.csv")

---
# Part 5: 추론 서버용 모델 내보내기

In [ ]:
# 추론 서버 (팀원 노트북)으로 복사할 파일들
print("\n[추론 서버 배포용 파일]")
print(f"복사할 폴더: {MERGED_PATH}")
print("")
print("팀원 노트북에서 로드:")
print('```python')
print('from transformers import WhisperProcessor, WhisperForConditionalGeneration')
print('')
print('processor = WhisperProcessor.from_pretrained("./whisper-korean-merged")')
print('model = WhisperForConditionalGeneration.from_pretrained(')
print('    "./whisper-korean-merged",')
print('    torch_dtype=torch.float16,')
print('    device_map="auto"')
print(')')
print('```')